<center><h1  style="color:white; background-color:#000000; border-radius: 0px; padding:25px;"> Fairness </h1></center>

This notebook illustrates fairness modeule on iris data set.

In [ ]:
import sys

sys.path.insert(0, "..")
sys.path.insert(0, "../data")

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from sklearn.metrics import average_precision_score, accuracy_score, roc_auc_score


from sklearn.pipeline import Pipeline

from woodtapper.fairness import OTFairBoostClassifier
from woodtapper.fairness.metrics import derive_fairness_metrics
from utils_data import make_stratify_labels, make_preprocessor_adult

## Load data :

In [ ]:
adult = fetch_openml("adult", version=2)
X_df = adult.data.drop(columns=["sex"])
numeric_cols = X_df.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X_df.columns if c not in numeric_cols]
X = X_df#.to_numpy()
y = adult.target
sensitive_attribute = adult.data[["sex"]]

In [ ]:
sensitive_attribute = adult.data["sex"]
sensitive_attribute = sensitive_attribute.map({"Male": 0, "Female": 1})
#sensitive_attribute = sensitive_attribute.to_numpy()
sensitive_attribute = pd.DataFrame(sensitive_attribute, dtype="int")
sensitive_attribute

In [ ]:
y = adult.target
y = y.map({"<=50K": 0, ">50K": 1})
#y = y.to_numpy()
y = pd.DataFrame(y, dtype="int")
y

In [ ]:
X_train, X_test, y_train, y_test, sensitive_attribute_train, sensitive_attribute_test = train_test_split(X, y, sensitive_attribute, random_state=0, test_size=0.2,
                                                                                                          stratify=make_stratify_labels(y, sensitive_attribute))

## Baseline LGBM:

In [ ]:
preprocessor = make_preprocessor_adult(numeric_cols, categorical_cols)

In [ ]:
baseline_lgbm = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    num_leaves=64,
    min_child_samples=50,
    colsample_bytree=0.9,
    subsample=0.9,
    subsample_freq=1,
    random_state=42,
    n_jobs=8,
    verbosity=-1
)
baseline_lgbm_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", baseline_lgbm)])
baseline_lgbm_pipeline.fit(X_train, y_train)

## OTFairBoostClassifier

In [ ]:
ot_clf = OTFairBoostClassifier(
    n_estimators=1000,
    lambda_fairness_value=1.0,
    fairness_mode="Demographic_Parity", #Demographic_Parity Equality_of_Odds
    verbosity=-1,
    learning_rate=0.1,
    num_leaves=64,
    min_child_samples=50,
    colsample_bytree=0.9,
    subsample=0.9,
    subsample_freq=1,
    random_state=42,
    n_jobs=8,
)

ot_model_pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("ot_clf", ot_clf)])
ot_model_pipeline.fit(X_train, y_train, ot_clf__sensitive_attribute=sensitive_attribute_train)

## Results analysis:

In [ ]:
models = {
    "Baseline LGBM": baseline_lgbm_pipeline,
    "OTFairBoost (Demographic Parity)": ot_model_pipeline,
}

results = []
group_rates = []
for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    (dp_gap, di, eopp_gap, fpr_gap, ppv_gap, dp_ovr_gap, 
     eopp_ovr_gap, fpr_ovr_gap, ppv_ovr_gap, rates_df) = derive_fairness_metrics(
        y_true=y_test,
        y_preds=y_pred,
        sensitive_attribute=sensitive_attribute_test,
        min_group_n=10,
    )

    results.append({
        "Model": model_name,
        "PR-AUC": average_precision_score(y_true=y_test, y_score=y_score),
        "ROC-AUC": roc_auc_score(y_true=y_test, y_score=y_score),
        "DP": dp_gap,
        "DI": di,
    })
    group_rates.append(rates_df.assign(Model=model_name))

comparison_df = pd.DataFrame(results).set_index("Model")
display(
    comparison_df.style
    .format(precision=3,)
)